In [1]:
# !pip install -q "transformers>=4.51.0" "datasets>=2.19.0" "accelerate>=0.30.0" "lm-eval>=0.4.8" pyyaml matplotlib

# import os
# REPO_URL = "https://github.com/syedmoazzamrazagilani/ATML-PA3.git"
# REPO_DIR = "/kaggle/working/repo"

# if not os.path.exists(REPO_DIR):
#     !git clone {REPO_URL} {REPO_DIR}
# else:
#     !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

# %cd {REPO_DIR}
# !wget -q -O first_plot_questions.yaml https://raw.githubusercontent.com/emergent-misalignment/emergent-misalignment/main/evaluation/first_plot_questions.yaml
# !rm -rf /kaggle/working/task1_outputs /kaggle/working/task1_smoke
# !mkdir -p /kaggle/working/task1_outputs

In [2]:
# import subprocess, shutil

# subprocess.run([
#     "python", "-m", "task1.run_task1",
#     "--stage", "smoke",
#     "--variant", "all",
#     "--output-dir", "/kaggle/working/task1_smoke",
#     "--yaml-path", "first_plot_questions.yaml",
# ], check=True)

# shutil.rmtree("/kaggle/working/task1_smoke", ignore_errors=True)
# print("\nSmoke test passed 100% cleanly! Starting full Task 1 pipeline.\n")

In [3]:
# import subprocess

# subprocess.run([
#     "python", "-m", "task1.run_task1",
#     "--stage", "prep",
#     "--output-dir", "/kaggle/working/task1_outputs",
#     "--yaml-path", "first_plot_questions.yaml",
# ], check=True)

In [4]:
# import subprocess

# subprocess.run([
#     "python", "-m", "task1.run_task1",
#     "--stage", "train",
#     "--variant", "all",
#     "--output-dir", "/kaggle/working/task1_outputs",
#     "--prune-optimizer-states",
# ], check=True)

In [5]:
# import subprocess

# subprocess.run([
#     "python", "-m", "task1.run_task1",
#     "--stage", "eval_and_diag",
#     "--output-dir", "/kaggle/working/task1_outputs",
#     "--yaml-path", "first_plot_questions.yaml",
# ], check=True)

In [6]:
# import os
# import shutil
# from pathlib import Path
# from IPython.display import FileLink, display

# out_dir = Path("/kaggle/working/task1_outputs")
# bundle_dir = Path("/kaggle/working/task1_report_bundle")
# shutil.rmtree(bundle_dir, ignore_errors=True)
# bundle_dir.mkdir(parents=True, exist_ok=True)

# for sub in ["eval", "figures"]:
#     if (out_dir / sub).exists():
#         shutil.copytree(out_dir / sub, bundle_dir / sub, dirs_exist_ok=True)

# for fname in ["data_selection_stats.json", "probe_ids.json"]:
#     if (out_dir / fname).exists():
#         shutil.copy2(out_dir / fname, bundle_dir / fname)

# for var in ["full_ft", "l2_sp", "lora_4", "lora_16", "oplora_16"]:
#     var_src = out_dir / var
#     var_dst = bundle_dir / var
#     if var_src.exists():
#         var_dst.mkdir(parents=True, exist_ok=True)
#         for log_name in ["train_log.jsonl", "run_summary.json", "merged_logit_verification.json"]:
#             if (var_src / log_name).exists():
#                 shutil.copy2(var_src / log_name, var_dst / log_name)
#         ckpt_src = var_src / "checkpoints"
#         if ckpt_src.exists():
#             for pt_file in ckpt_src.glob("adapter_update_*.pt"):
#                 (var_dst / "checkpoints").mkdir(parents=True, exist_ok=True)
#                 shutil.copy2(pt_file, var_dst / "checkpoints" / pt_file.name)

# os.chdir("/kaggle/working")
# zip_base = "/kaggle/working/task1_report_artifacts"
# shutil.make_archive(zip_base, "zip", bundle_dir)
# shutil.rmtree(bundle_dir, ignore_errors=True)

# zip_file = Path("task1_report_artifacts.zip")
# size_mb = zip_file.stat().st_size / (1024 ** 2)
# print("\n" + "=" * 60)
# print(f"Task 1 artifacts packaged: {zip_file.name} ({size_mb:.2f} MB)")
# print("=" * 60 + "\n")
# display(FileLink("task1_report_artifacts.zip"))

In [7]:
!pip install -q "transformers>=4.51.0" "datasets>=2.19.0" "accelerate>=0.30.0" "lm-eval>=0.4.8" pyyaml matplotlib

import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/syedmoazzamrazagilani/ATML-PA3.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

%cd {REPO_DIR}
!wget -q -O first_plot_questions.yaml https://raw.githubusercontent.com/emergent-misalignment/emergent-misalignment/main/evaluation/first_plot_questions.yaml

input_root = Path("/kaggle/input")
matches = list(input_root.rglob("metamath_40k.pt"))
assert matches, "Could not find metamath_40k.pt in /kaggle/input! Did you attach the previous notebook's output?"
prev_out_dir = matches[0].parent
print(f"Found previous run outputs at: {prev_out_dir}")

target_dir = Path("/kaggle/working/task1_outputs")
target_dir.mkdir(parents=True, exist_ok=True)

for fname in ["metamath_40k.pt", "probe_ids.json", "data_selection_stats.json"]:
    if (prev_out_dir / fname).exists():
        shutil.copy2(prev_out_dir / fname, target_dir / fname)

for var in ["full_ft", "l2_sp", "lora_4", "lora_16"]:
    src_var = prev_out_dir / var
    dst_var = target_dir / var
    assert (src_var / "run_summary.json").exists(), f"{var} is missing run_summary.json!"
    dst_var.mkdir(parents=True, exist_ok=True)
    for log_file in ["run_summary.json", "train_log.jsonl", "merged_logit_verification.json"]:
        if (src_var / log_file).exists():
            shutil.copy2(src_var / log_file, dst_var / log_file)

    dst_ckpt = dst_var / "checkpoints"
    dst_ckpt.mkdir(parents=True, exist_ok=True)
    if var in ("full_ft", "l2_sp"):
        for pt in (src_var / "checkpoints").glob("checkpoint_update_*.pt"):
            shutil.copy2(pt, dst_ckpt / pt.name)
    else:
        for pt in (src_var / "checkpoints").glob("adapter_update_*.pt"):
            shutil.copy2(pt, dst_ckpt / pt.name)

print("Successfully restored all 4 completed variants!")
!du -sh /kaggle/working/task1_outputs/*

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 91 (delta 46), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 51.22 KiB | 3.66 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/kaggle/working/repo
Found previous run outputs at: /kaggle/input/datasets/syedmoazzamraza/atml-pa3-task1-part1/task1_outputs
Successfully restored all 4 completed variants!
4.0K	/kaggle/working/task1_outputs/data_selection_stats.json
2.3G	/kaggle/working/task1_outputs/full_ft
1.2G	/kaggle/working/task1_outputs/l2_sp
156M	/kaggle/working/task1_outputs/lora_16
40M	/kaggle/working/ta

In [8]:
import subprocess

subprocess.run([
    "python", "-m", "task1.run_task1",
    "--stage", "train",
    "--variant", "oplora_16",
    "--output-dir", "/kaggle/working/task1_outputs",
    "--prune-optimizer-states",
], check=True)


--- Training Variant: oplora_16 (1250 updates) ---


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 453.77it/s]


CompletedProcess(args=['python', '-m', 'task1.run_task1', '--stage', 'train', '--variant', 'oplora_16', '--output-dir', '/kaggle/working/task1_outputs', '--prune-optimizer-states'], returncode=0)

In [9]:
import subprocess

subprocess.run([
    "python", "-m", "task1.run_task1",
    "--stage", "eval_and_diag",
    "--eval-batch-size", "16",
    "--output-dir", "/kaggle/working/task1_outputs",
    "--yaml-path", "first_plot_questions.yaml",
], check=True)

Generating validation split: 100%|██████████| 10042/10042 [00:00<00:00, 270809.04 examples/s]



--- Evaluating & Diagnosing Condition: baseline ---


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 505.55it/s]
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
generation_kwargs: {'do_sample': False, 'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Overwriting default num_fewshot of gsm8k_cot from 8 to 0
Map: 100%|██████████| 10042/10042 [00:01<00:00, 8683.35 examples/s]
[Task: boolq] metric acc is defined, but aggregation is not. using default aggregation=mean
[Task: boolq] metric acc is defined, but higher_is_better is not. using default higher_is_better=True
Generating validation split: 100%|██████████| 570/570 [00:00<00:00, 189082.04 examples/s]
Overwr


--- Evaluating & Diagnosing Condition: full_ft_mid ---


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 513.88it/s]
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
generation_kwargs: {'do_sample': False, 'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Overwriting default num_fewshot of gsm8k_cot from 8 to 0
Running generate_until requests: 100%|██████████| 1319/1319 [28:46<00:00,  1.31s/it]
[Task: boolq] metric acc is defined, but aggregation is not. using default aggregation=mean
[Task: boolq] metric acc is defined, but higher_is_better is not. using default higher_is_better=True
Overwriting default num_fewshot of hellaswag from None to 0
Overwriting default


--- Evaluating & Diagnosing Condition: full_ft ---


... (6 outputs left out)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 472.28it/s]
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
generation_kwargs: {'do_sample': False, 'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Overwriting default num_fewshot of gsm8k_cot from 8 to 0
Running generate_until requests: 100%|██████████| 1319/1319 [21:57<00:00,  1.00it/s]
[Task: boolq] metric acc is defined, but aggregation is not. using default aggregation=mean
[Task: boolq] metric acc is defined, but higher_is_better is not. using default higher_is_better=True
Overwriting default num_fewshot of hellaswag from None to 0
Overwriting default


--- Evaluating & Diagnosing Condition: oplora_16 ---


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 475.19it/s]
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
generation_kwargs: {'do_sample': False, 'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Overwriting default num_fewshot of gsm8k_cot from 8 to 0
Running generate_until requests: 100%|██████████| 1319/1319 [23:02<00:00,  1.05s/it]
[Task: boolq] metric acc is defined, but aggregation is not. using default aggregation=mean
[Task: boolq] metric acc is defined, but higher_is_better is not. using default higher_is_better=True
Overwriting default num_fewshot of hellaswag from None to 0
Overwriting default

CompletedProcess(args=['python', '-m', 'task1.run_task1', '--stage', 'eval_and_diag', '--eval-batch-size', '16', '--output-dir', '/kaggle/working/task1_outputs', '--yaml-path', 'first_plot_questions.yaml'], returncode=0)

In [10]:
import os
import shutil
from pathlib import Path
from IPython.display import FileLink, display

out_dir = Path("/kaggle/working/task1_outputs")
bundle_dir = Path("/kaggle/working/task1_report_bundle")
shutil.rmtree(bundle_dir, ignore_errors=True)
bundle_dir.mkdir(parents=True, exist_ok=True)

for sub in ["eval", "figures"]:
    if (out_dir / sub).exists():
        shutil.copytree(out_dir / sub, bundle_dir / sub, dirs_exist_ok=True)

for fname in ["data_selection_stats.json", "probe_ids.json"]:
    if (out_dir / fname).exists():
        shutil.copy2(out_dir / fname, bundle_dir / fname)

for var in ["full_ft", "l2_sp", "lora_4", "lora_16", "oplora_16"]:
    var_src = out_dir / var
    var_dst = bundle_dir / var
    if var_src.exists():
        var_dst.mkdir(parents=True, exist_ok=True)
        for log_name in ["train_log.jsonl", "run_summary.json", "merged_logit_verification.json"]:
            if (var_src / log_name).exists():
                shutil.copy2(var_src / log_name, var_dst / log_name)
        ckpt_src = var_src / "checkpoints"
        if ckpt_src.exists():
            for pt_file in ckpt_src.glob("adapter_update_*.pt"):
                (var_dst / "checkpoints").mkdir(parents=True, exist_ok=True)
                shutil.copy2(pt_file, var_dst / "checkpoints" / pt_file.name)

os.chdir("/kaggle/working")
zip_base = "/kaggle/working/task1_report_artifacts"
shutil.make_archive(zip_base, "zip", bundle_dir)
shutil.rmtree(bundle_dir, ignore_errors=True)

zip_file = Path("task1_report_artifacts.zip")
size_mb = zip_file.stat().st_size / (1024 ** 2)
print("\n" + "=" * 60)
print(f"Task 1 artifacts packaged: {zip_file.name} ({size_mb:.2f} MB)")
print("=" * 60 + "\n")
display(FileLink("task1_report_artifacts.zip"))


Task 1 artifacts packaged: task1_report_artifacts.zip (379.66 MB)



/kaggle/working/task1_report_artifacts.zip